# Cameo Requirements Extraction + Tag Update — Chained Jobs

A recipe for chaining jobs on the Istari Digital Platform from Python: register a Cameo `.mdzip` file as a Model, run an extraction job to read requirements, then chain a tag update job to write `Part number selected = PN12345` back into the model.

It uses [`istari_fluent`](../fluent), an opinionated, chainable wrapper over the official [`istari-digital-client`](https://docs.istaridigital.com/developers/SDK/01-setup). The concepts (Systems, Models, Jobs, Products, Resources) are identical to the platform UI &mdash; `istari_fluent` just packages them behind entity-oriented methods.

### What we cover

- Connecting to the platform with a Personal Access Token.
- Registering a Cameo `.mdzip` file as a Model.
- Running a first extraction job and reading the requirements output.
- Chaining a second job to update a tag on a requirement using `@istari:update_tags`.
- Tracing the backward lineage of the final result.

### Prerequisites

- An **Istari Digital Platform account** and a **Personal Access Token**. See the [Sign-up Guide](https://docs.istaridigital.com/users/account/sign-up) and [Personal Access Tokens](https://docs.istaridigital.com/users/user-guide/settings#developer-settings--personal-access-tokens).
- An agent with the **Cameo** integration and access to `@istari:extract` and `@istari:update_tags`. See [Manage Tool Access](https://docs.istaridigital.com/users/admin-guide/user-management#manage-tool-access-for-a-user).
- A Cameo `.mdzip` requirements file to extract from.

### 1 &middot; Credentials

Create a `.env` file **next to this notebook** with the same two variable names the official Python client uses:

```
ISTARI_REGISTRY_URL=https://...paste your platform's registry URL here...
ISTARI_PERSONAL_ACCESS_TOKEN=...paste your token here...
```

You can find both in the platform under **Settings &rarr; Developer Settings**. Treat the token as a secret &mdash; it grants API access as you. Don't commit it, don't paste it into chat, don't screenshot it.

> **On a corporate network with an internal CA?** Pass the bundle into `from_env()` (or set `ISTARI_CA_BUNDLE` in your `.env`). See the commented example in the connect cell below.

### 2 &middot; Install dependencies

From the repository root:

```bash
cd fluent
uv sync --extra experiment
```

This creates `fluent/.venv/` containing `istari_fluent`, `jupyter`, and `ipykernel`.

### 3 &middot; Register the venv as a Jupyter kernel

A bare virtualenv is not auto-discovered by VS Code / Cursor. Register it once:

```bash
# still inside fluent/
uv run python -m ipykernel install --user --name istari-fluent --display-name "Python (istari_fluent)"
```

Then reload this notebook's kernel picker (top-right) and select **"Python (istari_fluent)"**.

To remove the kernel later: `jupyter kernelspec uninstall istari-fluent`.

> **A note on `istari_fluent`** &mdash; this is a productivity layer maintained alongside the official SDK. It is not the officially supported client. For production integrations, keep the core [`istari-digital-client`](https://docs.istaridigital.com/developers/SDK/01-setup) as your source of truth; use `istari_fluent` to prototype, explore, and build notebooks faster.

## 1 &middot; Connect and verify

`IstariPlatform.from_env()` reads `ISTARI_REGISTRY_URL` and `ISTARI_PERSONAL_ACCESS_TOKEN` and returns an `IstariPlatform` object.

After connecting we run a quick **readiness check**: one round-trip that confirms the platform is reachable and your token is accepted.

In [33]:
from pathlib import Path

from istari_fluent import IstariPlatform, JobDefinition

platform = IstariPlatform.from_env()

# Corporate network with an internal CA bundle? Point from_env() at your .pem:
# platform = IstariPlatform.from_env(ca_bundle="/path/to/ca.pem")
#
# Or set ISTARI_CA_BUNDLE in your .env / shell and just call from_env() above.

# Readiness check: one round-trip to confirm the platform answers and the token is accepted.
report = platform.client.readiness_check()
assert report.healthy, f"Platform reports unhealthy: {report}"

print(platform)

/Users/craighahn/Documents/GitHub/cookbook/istari-digital-client-cookbook/fluent/.venv/lib/python3.11/site-packages/istari_digital_client/log_utils.py:32: UserWarning: SDK is incompatible with Istari Registry v10.12.22 (affected APIs: v2_main:Job, v2_main:Module). Please update your SDK to match the server version.
  result = func(*args, **kwargs)
2026-05-06 22:05:50 -  istari_digital_client.compatibility:process_response_headers:81 - WARNING - SDK is incompatible with Istari Registry v10.12.22 (affected APIs: v2_main:Job, v2_main:Module). Please update your SDK to match the server version.
2026-05-06 22:05:50 -  istari_digital_client.compatibility:process_response_headers:81 - WARNING - SDK is incompatible with Istari Registry v10.12.22 (affected APIs: v2_main:Job, v2_main:Module). Please update your SDK to match the server version.
2026-05-06 22:05:50 -  istari_digital_client.compatibility:process_response_headers:81 - WARNING - SDK is incompatible with Istari Registry v10.12.22 (aff

IstariPlatform connected to https://fileservice-v2.demo.istari.app


## 2 &middot; Configure job parameters

Set the path to your Cameo `.mdzip` file and the target Cameo version and OS.
Adjust `TOOL_VERSION` and `OPERATING_SYSTEM` to match your agent's configuration.

To see available functions and supported tool versions on your platform:
```python
functions = platform.client.list_functions(tool="dassault_cameo")
for f in functions.items:
    print(f.name, f.tool_versions, f.operating_systems)
```

In [35]:
# --- Configure these for your environment ---
MDZIP_PATH = Path.cwd() / "NCXTable-example.mdzip"  # path to your Cameo file
TOOL_VERSION = "2024x-refresh2"                             # adjust to your Cameo version
OPERATING_SYSTEM = "Windows 11"                             # adjust to your agent's OS
DISPLAY_NAME = "Cameo Requirements Model (extraction) NCXTable"
EXTERNAL_ID = "cameo-requirements-extraction-tutorial-ncxtable"
# --------------------------------------------

assert MDZIP_PATH.exists(), f"Cameo file not found: {MDZIP_PATH}"
print(f"Using model file: {MDZIP_PATH}")
print(f"Tool version:     {TOOL_VERSION}")
print(f"Operating system: {OPERATING_SYSTEM}")

Using model file: /Users/craighahn/Documents/GitHub/cookbook/istari-digital-client-cookbook/samples/NCXTable-example.mdzip
Tool version:     2024x-refresh2
Operating system: Windows 11


## 3 &middot; Register the Cameo file as a Model Resource

In Istari terminology, registering a local file creates a **Resource** of type **model**. Each resource has a stable identity with a **revision** history.

We also tag the resource with an `external_id` for reference.

In [37]:
model = platform.upload_model(
    MDZIP_PATH,
    external_id=EXTERNAL_ID,
    display_name=DISPLAY_NAME,
)
print(f"Uploaded new model {model.id}")

print(model)

Uploaded new model 3e0e1be3-604e-41be-8efa-03fc5f751b26
Model('Cameo Requirements Model (extraction) NCXTable', filename='NCXTable-example.mdzip', id=3e0e1be3-604e-41be-8efa-03fc5f751b26, file=3595e757-c811-4bce-ab99-e63d22be10ab, rev=d4bc209b-dd17-419c-8a29-a8fa6ba490a6)


## 4 &middot; Run the first extraction job

A **Job** is an instruction to the platform to run a specific **function** from a **tool** against a Model revision. Here we run `@istari:extract` with the `dassault_cameo` tool.

We use the two-step pattern so you can watch the status evolve:

1. `model.submit_job(definition)` returns a `JobView` immediately &mdash; the job is queued but not yet finished.
2. `job.wait(on_poll=...)` blocks until the job reaches a terminal state, calling the `on_poll` callback on every poll (default every 5 s).
3. `.on_success()` raises `RuntimeError` if the job ended in `FAILED`.

> **Shortcut:** `model.run_job(definition)` submits + waits + checks success in one call.

In [38]:
extract = JobDefinition(
    function="@istari:extract",
    tool_name="dassault_cameo",
    #tool_version=TOOL_VERSION,
    #operating_system=OPERATING_SYSTEM,
)

job1 = model.submit_job(extract)
print(f"Submitted job {job1.id}; polling...")

job1.wait(
    timeout=600,
    on_poll=lambda j: print(f"  [{j.status}] id={j.id}"),
).on_success()

print(f"\nJob 1 finished: {job1.status}")

# One-liner alternative: model.run_job(extract, timeout=600, on_poll=print)
# `run_job` == `submit_job` + `wait().on_success()`, with the same on_poll hook.

Submitted job 8d3a40e1-9662-4f7d-bde4-fe7bb615938e; polling...
  [Pending] id=8d3a40e1-9662-4f7d-bde4-fe7bb615938e
  [Running] id=8d3a40e1-9662-4f7d-bde4-fe7bb615938e
  [Running] id=8d3a40e1-9662-4f7d-bde4-fe7bb615938e
  [Running] id=8d3a40e1-9662-4f7d-bde4-fe7bb615938e
  [Running] id=8d3a40e1-9662-4f7d-bde4-fe7bb615938e
  [Running] id=8d3a40e1-9662-4f7d-bde4-fe7bb615938e
  [Running] id=8d3a40e1-9662-4f7d-bde4-fe7bb615938e
  [Running] id=8d3a40e1-9662-4f7d-bde4-fe7bb615938e
  [Running] id=8d3a40e1-9662-4f7d-bde4-fe7bb615938e
  [Running] id=8d3a40e1-9662-4f7d-bde4-fe7bb615938e
  [Completed] id=8d3a40e1-9662-4f7d-bde4-fe7bb615938e

Job 1 finished: Completed


## 5 &middot; Inspect the products

Every job records the exact artifact revisions it wrote. `job1.get_products()` returns a list of `ResourceView` objects **pinned to those revisions**.

Every product has a `file_id` **and** a `rev_id`. The file id is the stable identity of the artifact; the revision id is the specific version this job produced.

In [39]:
products_1 = job1.get_products()
print(f"Job 1 wrote {len(products_1)} product(s):\n")
for p in products_1:
    print(f"  - {p.type:10s}  name={p.name!r:40s}  file={p.file_id}  rev={p.revision_id}")

Job 1 wrote 8 product(s):

  - Artifact    name='istari-module-stdout.txt'                file=a690e14e-8671-4580-b4bb-74745878c009  rev=6f484dc3-8223-4d99-b58e-f7fda4fdda37
  - Artifact    name='istari-module-stderr.txt'                file=266077ca-cd80-4ef7-9aac-3c52042d7f6c  rev=dd6bb28b-08d9-45b4-8676-ca1305acd0ae
  - Artifact    name='istari-agent-module-exit-status.txt'     file=4cf827ac-bdd1-4e70-8b16-426f00e81c43  rev=2bcc165b-0b0a-446b-81cc-87cf20190dd7
  - Artifact    name='requirements.json'                       file=bf9d7b95-879a-4fe2-9787-4aaa5454be60  rev=e8af0e0f-8672-45f9-896c-87e13beb1383
  - Artifact    name='blocks.json'                             file=d6a00635-7b00-4bed-a3c7-6b2af0475fd5  rev=a17a9436-421e-42a6-945f-2c3ece31088f
  - Artifact    name='other_elements.json'                     file=8e2f1b92-247f-4bc4-b45b-3ce5a7b859d6  rev=b6db326d-2ccc-4f72-b63c-9126e5d660f8
  - Artifact    name='relationships.json'                      file=e4ceaa97-72ad-4900-8242

## 6 &middot; Read requirements and pick one to update

Load `requirements.json` from Job 1 and inspect the requirements to find the element you want to update.

Each requirement object contains an `id` field (the internal Cameo element ID) and a `name` field.
We'll use the `element_id` when calling `update_tags` for precise targeting — this avoids ambiguity
if multiple requirements share a similar name.

Adjust `TARGET_REQUIREMENT_NAME` below to match the requirement you want to update.

In [40]:
TARGET_REQUIREMENT_NAME = "Requirement REQ-001 Plate Thickness"  # adjust to match a requirement in your model

requirements_artifact = job1.find_product(filename="requirements.json")
assert requirements_artifact is not None, (
    "requirements.json not found. "
    f"Available: {[p.name for p in products_1]}"
)

requirements_data = requirements_artifact.read_json()
print(f"Found {len(requirements_data)} requirements\n")

# Print all requirements so you can pick the right one
for req in requirements_data:
    print(f"  id={req['id']}")
    print(f"  name={req['name']}")
    print(f"  tags={req.get('tags', {})}")
    print()

# Find the target requirement by name
target_req = next(
    (r for r in requirements_data if r["name"] == TARGET_REQUIREMENT_NAME),
    None
)
assert target_req is not None, (
    f"Requirement {TARGET_REQUIREMENT_NAME!r} not found. "
    f"Available names: {[r['name'] for r in requirements_data]}"
)

TARGET_ELEMENT_ID = target_req["id"]
print(f"\nTarget requirement:")
print(f"  name: {target_req['name']}")
print(f"  id:   {TARGET_ELEMENT_ID}")
print(f"  existing tags: {target_req.get('tags', {})}")

Found 8 requirements

  id=_2021x_2_38b206bc_1744829738919_924302_3837
  name=Requirement REQ-001 Plate Thickness
  tags={'Text': 'The base plate (PLATE_1) thickness shall be between 26 mm and 30 mm to provide adequate bending stiffness while remaining within the prescribed mass budget for the test assembly.', 'Id': '1545599', 'TBD/TBR': False}

  id=_2021x_2_38b206bc_1744829804820_164896_3864
  name=Requirement REQ-002 Plate Length
  tags={'Text': 'The base plate (PLATE_1) length along the X axis shall be between 120 mm and 160 mm to fit within the allocated structural bay.', 'Id': '1545600', 'TBD/TBR': False}

  id=_2022x_2_24f90527_1750033427003_556537_3365
  name=Requirement REQ-003 Plate Width
  tags={'Text': 'The base plate (PLATE_1) width along the Y axis shall be between 120 mm and 160 mm to match the length constraint and maintain a square footprint for symmetric loading.', 'Id': '1545601', 'TBD/TBR': False}

  id=_2024x_2_1fd504ba_1777994538882_283919_2916
  name=Requirement 

## 7 &middot; Chain a second job — update the tag

Now we run `@istari:update_tags` on the **original `.mdzip` model** to write the part number
`PN12345` back into the the requirement in the Cameo model.

Key points from the Cameo integration docs:

- `element_id` targets the exact element (preferred over `element_name` to avoid ambiguity)
- `replace_existing: true` overwrites any existing value for that tag
- The result is a **new revision of the same model** — the updated `.mdzip` is committed back
  to the platform as a new model version, not produced as a separate artifact

The `update_tags` job runs on `model` (the original `.mdzip`), not on the JSON artifact from Job 1.

In [41]:
updates = [
    {
        "element_id": "_2021x_2_38b206bc_1744829738919_924302_3837",
        "replace_existing": False,
        "tags": {
            "Text": "Selected part number: PN12345"
        }
    }
]

update_tags = JobDefinition(
    function="@istari:update_tags",
    tool_name="dassault_cameo",
    #tool_version=TOOL_VERSION,
    #operating_system=OPERATING_SYSTEM,
    parameters={"updates": updates},
)

job2 = model.submit_job(update_tags)
print(f"Submitted tag update job {job2.id}; polling...")

job2.wait(
    timeout=600,
    on_poll=lambda j: print(f"  [{j.status}] id={j.id}"),
).on_success()

print(f"\nJob 2 finished: {job2.status}")

products_2 = job2.get_products()
print(f"\nJob 2 wrote {len(products_2)} product(s):")
for p in products_2:
    print(f"  - {p.type:10s}  name={p.name!r:40s}  rev={p.revision_id}")

Submitted tag update job 6d6c5cf3-298b-4452-9546-1873588550d8; polling...
  [Pending] id=6d6c5cf3-298b-4452-9546-1873588550d8
  [Validating] id=6d6c5cf3-298b-4452-9546-1873588550d8
  [Running] id=6d6c5cf3-298b-4452-9546-1873588550d8
  [Running] id=6d6c5cf3-298b-4452-9546-1873588550d8
  [Running] id=6d6c5cf3-298b-4452-9546-1873588550d8
  [Running] id=6d6c5cf3-298b-4452-9546-1873588550d8
  [Running] id=6d6c5cf3-298b-4452-9546-1873588550d8
  [Running] id=6d6c5cf3-298b-4452-9546-1873588550d8
  [Running] id=6d6c5cf3-298b-4452-9546-1873588550d8
  [Running] id=6d6c5cf3-298b-4452-9546-1873588550d8
  [Completed] id=6d6c5cf3-298b-4452-9546-1873588550d8

Job 2 finished: Completed

Job 2 wrote 4 product(s):
  - Artifact    name='istari-module-stdout.txt'                rev=12d225ac-c668-42ff-b4be-7009726135a6
  - Artifact    name='istari-agent-module-exit-status.txt'     rev=5515fb9b-1528-4be3-901f-4432fc378e8c
  - Artifact    name='istari-module-stderr.txt'                rev=87d31b78-3114-426b-8

## 8 &middot; Verify the update

Per the Cameo integration docs, `update_tags` commits the changes back to the platform as a **new
version of the same model**. To confirm the tag was written correctly, we fetch the updated model
and run a fresh extraction to read the tags back out.

This also demonstrates the full digital thread: extract → update → re-extract to verify.

In [42]:
# Re-run extraction on the updated model to verify the tag was written
job3 = model.submit_job(extract)
print(f"Submitted verification extraction {job3.id}; polling...")

job3.wait(
    timeout=600,
    on_poll=lambda j: print(f"  [{j.status}] id={j.id}"),
).on_success()

print(f"\nVerification job finished: {job3.status}")

# Read the updated requirements
updated_requirements_artifact = job3.find_product(filename="requirements.json")
assert updated_requirements_artifact is not None

updated_requirements = updated_requirements_artifact.read_json()

# Find our target requirement and check its tags
updated_req = next(
    (r for r in updated_requirements if r["id"] == TARGET_ELEMENT_ID),
    None
)
assert updated_req is not None, "Could not find target requirement in updated extraction"

print(f"\nUpdated requirement:")
print(f"  name: {updated_req['name']}")
print(f"  tags: {updated_req.get('tags', {})}")

#assert updated_req.get("tags", {}).get("Part number selected") == "PN12345", (
 #   f"Tag not updated as expected. Got: {updated_req.get('tags', {})}"
#)
#print("\n✓ Tag 'Part number selected' = 'PN12345' confirmed in updated model")

Submitted verification extraction d6d106ed-7217-456e-908a-e5747fad77c4; polling...
  [Pending] id=d6d106ed-7217-456e-908a-e5747fad77c4
  [Claimed] id=d6d106ed-7217-456e-908a-e5747fad77c4
  [Running] id=d6d106ed-7217-456e-908a-e5747fad77c4
  [Running] id=d6d106ed-7217-456e-908a-e5747fad77c4
  [Running] id=d6d106ed-7217-456e-908a-e5747fad77c4
  [Running] id=d6d106ed-7217-456e-908a-e5747fad77c4
  [Running] id=d6d106ed-7217-456e-908a-e5747fad77c4
  [Running] id=d6d106ed-7217-456e-908a-e5747fad77c4
  [Running] id=d6d106ed-7217-456e-908a-e5747fad77c4
  [Running] id=d6d106ed-7217-456e-908a-e5747fad77c4
  [Uploading] id=d6d106ed-7217-456e-908a-e5747fad77c4
  [Completed] id=d6d106ed-7217-456e-908a-e5747fad77c4

Verification job finished: Completed

Updated requirement:
  name: Requirement REQ-001 Plate Thickness
  tags: {'Text': 'The base plate (PLATE_1) thickness shall be between 26 mm and 30 mm to provide adequate bending stiffness while remaining within the prescribed mass budget for the tes

## 9 &middot; Trace the lineage

The payoff of uploading, running jobs, and promoting artifacts is that **every revision knows how it got there**. `get_lineage()` walks backward from any revision and classifies each step:

| Step | Meaning |
|---|---|
| `upload` | A fresh file &mdash; no sources, the root of a chain |
| `job_run` | Produced by a Job |
| `promotion` | A Model promoted from another revision (relationship `promoted_from`) |
| `derived` | Any other derivation |

Below we trace `requirements.json` from the verification extraction back to the original upload.
The chain should show the full extract → update → re-extract sequence.

In [43]:
tree = updated_requirements_artifact.get_lineage(max_depth=8)
print("Lineage for verification requirements.json:\n")
tree.print_tree()

Lineage for verification requirements.json:

- Artifact 'requirements.json' (rev=adf40f28-088c-4092-8daf-99187e9d4c35)
    step=job_run  created=2026-05-07 02:17
  - Job '@istari:extract (d6d106ed-7217-456e-908a-e5747fad77c4)' (rev=89de2a70-7841-4484-8eac-5c92c5eb1693)  [via -]
      step=job_run  created=2026-05-07 02:16
    - Model 'NCXTable-example.mdzip' (rev=dcf19210-ee39-4961-9195-e988add6bbb3)  [via -]
        step=job_run  created=2026-05-07 02:14
      - Job '@istari:update_tags (6d6c5cf3-298b-4452-9546-1873588550d8)' (rev=d197d08f-e6e0-45fb-928b-95f683e7124a)  [via -]
          step=job_run  created=2026-05-07 02:14
        - Model 'Cameo Requirements Model (extraction) NCXTable' (rev=d4bc209b-dd17-419c-8a29-a8fa6ba490a6)  [via -]
            step=upload  created=2026-05-07 02:07


## Verify in the UI

Sign in to the same platform you used for your token and cross-check:

1. **Files / Models** &mdash; You should see the model with the display name set in Step 2.
2. **Jobs / Activity** &mdash; Three jobs for `dassault_cameo`: two `@istari:extract` and one `@istari:update_tags`.
3. **Resources** &mdash; On the update_tags job, confirm the updated model revision was produced.
4. **Verification** &mdash; On the third job's resources, confirm `requirements.json` shows the part number `PN12345` listed in the target requirement.

## Optional &middot; Archive the model

Archiving hides the Model from default listings in the UI and API. The data and lineage stay intact &mdash; this is a soft delete you can reverse later. Run this when you're done experimenting and don't want the tutorial Model cluttering your workspace.

In [ ]:
model.archive()